In [ ]:
import os
   
docs_directory = "../data/text_files"
raw_docs = []
filenames = []
metadatas = []
   
for filename in os.listdir(docs_directory):
        if filename.endswith(".txt"):
            path = os.path.join(docs_directory, filename)
            with open(path, "r", encoding='utf-8') as f :
                text = f.read()
                raw_docs.append(text)
                filenames.append(filename)
                metadatas.append({"source":filename})

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
   
splitter = RecursiveCharacterTextSplitter(
        chunk_size = 600,
        chunk_overlap = 80,
        separators=["\n\n", "\n", ".", " ", ""]
    )
all_chunks = splitter.create_documents(raw_docs, metadatas=metadatas)
print(f"Nombre de chunks créés : {len(all_chunks)}")
print(f"Exemple de chunk : {all_chunks[0].page_content[:200]}...")

In [ ]:
# Les chunks sont déjà des Documents LangChain avec leurs métadonnées.
documents = all_chunks


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Modèle gratuit exécuté localement (aucune clé API nécessaire).
# Le téléchargement du modèle se fait lors de la première exécution.
emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)


In [ ]:
# Créer la base vectorielle Chroma et y enregistrer les chunks
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=emb,
    collection_name="support_docs",
    persist_directory="./vector_store",
)


In [ ]:
query = "Comment puis-je réinitialiser mon mot de passe ?"
results = vector_store.similarity_search(query, k=3)

print("Résultats de la recherche de similarité :")
for i, result in enumerate(results):
    print(f"Résultat {i+1} : {result.page_content[:200]}... (source: {result.metadata['source']})")